In [1]:
# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# project function imports used in notebook
# NOTE: requires PYTHONPATH to be set correctly
from nndl import vectorize_samples, plot_history

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

In [2]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
#dev = tf.config.list_physical_devices()
#print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
#dev = tf.config.list_logical_devices()
#print('Available Devices : ', dev)

# Chapter 8: Introduction to Deep Learning for Computer Vision

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

**Convolutional neural networks** also known as **convnets**, are introduced in this chapter. 
Computer vision is the earliest and biggest success story of deep learning.  
A type of deep learning model called convolutional
neural networks started getting remarkably good results on image classification
competitions in the early 2010's. In these sections we will look at what
convnets are and how to use them to solve computer vision and image
classification tasks.

## 8.1 Introduction to Convnets

Before we get into the details of what a convnet is and how it works, lets revisit the MNIST digits task.  We
looked at this task a couple of times in previous materials and saw a simple sequential densely connected
neural network that could get 97-98% test accuracy on MNIST.  Even though the following
convnet example will be basic, it will be able to easily outperform the densely connected sequential
network we used previously.

The following listing shows what a basic convnet looks like. It’s a stack of `Conv2D`
and `MaxPooling2D` layers.  We will discuss in a bit what those new layers are and how they work.
The model is using the Functional API to create a network that takes 28x28 shaped greyscale images
as input.  There are three convolutional layers with max pooling layers in between.  You will notice that
the `filters` parameter of the `Conv2d` layers increases from 32 to 64 to 128 as the network progresses. 
Also don't forget the `kernel_size` parameter, which is 3 for all layers, and will be explained in a moment.
Since this is a multi-class classification task, the final layer is a `Dense` layer with
10 output units using a `softmax` output activation function.

If you are paying attention, you may notice that the images are not flattened as previously, before going into
the first `Conv2D` layer.  But instead there is a `Flatten` layer that happens after the last `Conv2D`
layer for the `Dense` output layer.

In [3]:
# simple convolutional network for MNIST dataset
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.MaxPooling2D(pool_size=2)(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs=inputs, outputs=outputs)

As we hinted at, a convnet takes as input tensors of shape `(image_height, image_width, image_channels)`
(ignoring the first sample dimension), instead of a
simple vector of input features.  Thus the MNIST images get fed in as shape `(28, 28, 1)` since these images are greyscale
and only have 1 color channel of information.

The architecture of the convnet thus looks like this.  Notice the number of trainable parameters in the convolutional layers and
the max pooling layers.

In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        11,530 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 104,202 (407.04 KB)

 Trainable params: 104,202 (407.04 KB)

 Non-trainable params: 0 (0.00 B)

Notice that the output of every `Conv2D` and `MaxPooling2D` layer is also a rank-3 tensor of shape
`(height, width, channels)`.  The width and height dimensions tend to shrink as you go deeper
in the model (we will see in a moment why).  The number of channels is controlled by the first
argument to the `Conv2D` layers (32, 64, 128 respectively).  You can think of the "channels"
that happen in convolutional layers as the number of categories or features that the convolutional
layer is supposed to try and identify in its inputs.

After the last `Conv2D` layer, we end up with an output of shape `(3, 3, 128)` - a
3x3 feature map of 128 channels.  This output is flattened into a vector and fed
into a densely connected classifier like those you're already familiar with, which
here does a 10-way multi-class classification.

Now, let's train the convnet on the MNIST digits. Because we're doing 10-way
classification with softmax output, we'll use the categorical crossentropy loss, and because
our labels are integers, we'll use the sparse version, `sparse_categorical_crossentropy`.

In [5]:
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# perform some cleaning.  as before we turn the pixel
# greyscale values into integers ranging from 0 to 1.  We
# reshape the input to be shape (samples, 28, 28, 1) as expected
# by convnet for input
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype("float32") / 255

In [6]:
# compile the model for multi-class classification, and evaluate
# accuracy metric while training
model.compile(optimizer="rmsprop",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# fit model for 5 epochs of training
history = model.fit(train_images, train_labels, epochs=5, batch_size=64)

Epoch 1/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9489 - loss: 0.1651
Epoch 2/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 824us/step - accuracy: 0.9860 - loss: 0.0454
Epoch 3/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9904 - loss: 0.0322
Epoch 4/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9930 - loss: 0.0234
Epoch 5/5
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 903us/step - accuracy: 0.9940 - loss: 0.0189


You may notice the model quickly achieves 99+% accuracy on the data it is training with.
How does it evaluate on unseen test data?

In [7]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"Test accuracy: {test_acc:.3f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9905 - loss: 0.0305
Test accuracy: 0.990


Whereas the densely connected model we first discussed had a test accuracy of around 97-98%, the
basic convnet has a test accuracy of 99.1% or more.  We decreased the error rate by about 60% relative
to the previous performance!

Why does a simple convnet work so well compared to densely connected layers?  To answer, you need
to learn what the `Conv2D` and `MaxPooling2D` layers are and what they are doing.

### 8.1.1 The Convolution Operation

The fundamental difference between a densely connected layer and a convolution
layer is this: Dense layers learn global patterns in their input feature space (for example,
for a MNIST digit, patterns involving all pixels), whereas convolution layers learn
local patterns—in the case of images, patterns found in small 2D windows of the
inputs.  In the above example, since `kernel_size=3` for all convolutional layers,
these windows were all 3x3 shaped (3 pixels in width by 3 in height).

Images have a 2D structure.  When you flatten an image into a vector for a `Dense` layer,
you loose some information about that structure.  A `Conv2D` layer, as the name implies, works
on width x height local patterns, keeping the 2-dimensional structure information directly
accessible by these layers representations.


### 8.1.2 The Max-pooling Operation

The role of max pooling is to aggressively downsample feature maps, similar to strided convolutions.

Max pooling consists of extracting windows from the input feature map, conceptually similar to what convolutions
do.  But instead of a linear transformation (tensor product with the convolution kernel), the `max` operation
is simply applied to each window.

Why downsample feature maps this way? Why not remove the max-pooling layers
and keep fairly large feature maps all the way up? The first model, if we remove
max-pooling layers, would look like this:

In [8]:
inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(filters=32, kernel_size=3, activation="relu")(inputs)
x = layers.Conv2D(filters=64, kernel_size=3, activation="relu")(x)
x = layers.Conv2D(filters=128, kernel_size=3, activation="relu")(x)
x = layers.Flatten()(x)
outputs = layers.Dense(10, activation="softmax")(x)
model_no_max_pool = keras.Model(inputs=inputs, outputs=outputs)

In [9]:
model_no_max_pool.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 22, 22, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 61952)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │       619,530 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 712,202 (2.72 MB)

 Trainable params: 712,202 (2.72 MB)

 Non-trainable params: 0 (0.00 B)

What’s wrong with this setup? Two things:

1. It isn’t conducive to learning a spatial hierarchy of features. The 3 × 3 windows
   in the third layer will only contain information coming from 7 × 7 windows in
   the initial input. We need the features from the last convolution layer to contain
   information about the totality of the input.
2. The final feature map has 22 × 22 × 128 = 61,952 total coefficients per sample.
   This is huge. When you flatten it to stick a Dense layer of size 10 on top, that
   layer would have over half a million parameters. This is far too large for such a
   small model and would result in intense overfitting.

In short, the reason to use downsampling is to reduce the number of feature-map
coefficients to process, as well as to induce spatial-filter hierarchies by making successive
convolution layers look at increasingly large windows (in terms of the fraction of
the original input they cover).


## Summary

Some of the important things to keep in mind convnets

<font color='blue'>

- The visual world is fundamentally **translation-invariant**, convolutions learn translation-invariant kernels.
- The visual world is fundamentally **spatially hierarchical**, successive convolution layers are capable of learning increasingly high-level spatial hierarchies.
- Convolution layers operate by sliding a window to extract patches, and applies convolution kernel learned to each pass.
- Every dimension in the depth axis from a convolution is a **feature map**.
- The role of max pooling is to aggressively downsample feature maps.